In [1]:
"""
5-Times Sit-to-Stand (5STS)
=================================================
Clinical ground truth:
  >= 12 seconds to complete 5 full sit-to-stand cycles = HIGH fall risk

Training dataset: NWU Human Pose Dataset
  - 50,727 rows of OpenPose keypoints (18 joints × x, y, confidence)
  - Labels: 'sit' (0) and 'stand' (1)
  - Source: https://dayta.nwu.ac.za/articles/dataset/
            Human_pose_dataset_sit_stand_pose_classes_/23290937

Pipeline split:
  TRAINING  → NWU CSV (static OpenPose frames, no video needed)
  INFERENCE → MoveNet Lightning on patient webcam/video (real-time, home)

Why NWU for training + MoveNet for inference works:
  Both use the same COCO-compatible joint layout (OpenPose 18 ≈ COCO 17).
  We align them by mapping NWU's 18 joints down to the 17 COCO joints
  MoveNet uses, dropping the redundant mid-hip joint OpenPose adds.

Model: Conv1D → GRU → Dense(sigmoid)
  - Input : (SEQUENCE_LEN, 17 × 2) = (60, 34)
  - Output: probability of 'standing' state

Inference state machine:
  Counts 10 sit↔stand transitions = 5 complete reps,
  measures elapsed time, flags against 12s threshold.

Refinements applied (v2):
  1. Sit class oversampling  — tighter window step for sit (seq_len//6 vs seq_len//2)
                               brings sit:stand ratio from 1:2.9 → ~1:1
  2. Manual class weights    — {sit:3.0, stand:1.0} replacing formula weights
                               directly targets the observed Sit F1=0.83 gap
  3. Label smoothing (0.1)   — reduces overconfidence, closes train/val loss gap
  4. Tighter state machine   — STAND_THRESHOLD raised 0.60→0.65, SIT_THRESHOLD
                               lowered 0.40→0.35 to reduce false sit detections

Install:
  pip install tensorflow tensorflow-hub opencv-python numpy pandas tqdm scikit-learn
  pip install ipykernel openpyxl
"""

"\n5-Times Sit-to-Stand (5STS)\n=================================================\nClinical ground truth:\n  >= 12 seconds to complete 5 full sit-to-stand cycles = HIGH fall risk\n\nTraining dataset: NWU Human Pose Dataset\n  - 50,727 rows of OpenPose keypoints (18 joints × x, y, confidence)\n  - Labels: 'sit' (0) and 'stand' (1)\n  - Source: https://dayta.nwu.ac.za/articles/dataset/\n            Human_pose_dataset_sit_stand_pose_classes_/23290937\n\nPipeline split:\n  TRAINING  → NWU CSV (static OpenPose frames, no video needed)\n  INFERENCE → MoveNet Lightning on patient webcam/video (real-time, home)\n\nWhy NWU for training + MoveNet for inference works:\n  Both use the same COCO-compatible joint layout (OpenPose 18 ≈ COCO 17).\n  We align them by mapping NWU's 18 joints down to the 17 COCO joints\n  MoveNet uses, dropping the redundant mid-hip joint OpenPose adds.\n\nModel: Conv1D → GRU → Dense(sigmoid)\n  - Input : (SEQUENCE_LEN, 17 × 2) = (60, 34)\n  - Output: probability of 'sta

In [2]:
import sys
import os
import json
import random
import numpy as np
import pandas as pd
from datetime import datetime
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

sys.path.append(os.path.abspath("../src"))
from shared_utils import (
    load_movenet,
    extract_keypoints_from_video,
    normalize_keypoints,
    INPUT_DIM,      # 34  (17 joints × 2)
    KP,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

c:\Users\sherl\Downloads\ai_remote_health_monitoring\5_sit_to_stand_monitoring\.venv\Lib\site-packages\tensorflow_hub\__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [3]:
# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
NWU_XLSX_PATH = "../nwu_human_pose_dataset/dataset_HumanPose_KeypointCoordinates.xlsx"   # download from NWU dataset page

SEQUENCE_LEN = 60      # frames per synthetic window (~2s at 30fps)
BATCH_SIZE   = 32
EPOCHS       = 30
LR           = 1e-3

FALL_RISK_THRESHOLD = 12.0   # seconds
WORSEN_DELTA        = 2.0    # seconds increase → flag worsening

# NWU uses OpenPose 18 joints. We align to MoveNet COCO 17 by
# dropping joint index 1 (OpenPose "Neck" — not in COCO 17).
# All other 17 joints map 1:1 to COCO order after this drop.
NWU_NUM_JOINTS   = 18
NWU_DROP_JOINT   = 1    # OpenPose "Neck" — absent in COCO 17
NWU_ALIGNED_DIM  = 17 * 2   # 34  — matches INPUT_DIM after dropping Neck

In [4]:
# ─────────────────────────────────────────────
# STEP 1 — LOAD & PARSE NWU XLSX
# ─────────────────────────────────────────────
def load_nwu_dataset(xlsx_path: str) -> tuple:
    """
    Loads the NWU Human Pose Dataset XLSX.

    XLSX structure (57 columns):
      Filename,
      x0,y0,c0 ... x17,y17,c17    ← 18 OpenPose joints × 3
      class (number), class (text)

    Returns:
      X : (N, 34) float32  — aligned & normalised x,y (17 joints after drop)
      y : (N,)    int32    — 0=sit, 1=stand
    """
    print(f"[NWU] Loading: {xlsx_path}")
    df         = pd.read_excel(xlsx_path)
    df.columns = df.columns.str.strip()

    all_cols = df.columns.tolist()

    # ── Extract x,y columns by pattern, exclude confidence (c*) and metadata ──
    x_cols = [c for c in all_cols
              if c.strip().startswith(('x', ' x'))
              and 'class' not in c.lower()
              and 'Filename' not in c]
    y_cols = [c for c in all_cols
              if c.strip().startswith(('y', ' y'))
              and 'class' not in c.lower()
              and 'Filename' not in c]

    # Interleave into [x0,y0, x1,y1, ..., x17,y17]
    interleaved = []
    for xc, yc in zip(x_cols, y_cols):
        interleaved.extend([xc, yc])

    raw = df[interleaved].values.astype(np.float32)   # (N, 36)

    # ── Align to COCO 17: drop OpenPose Neck (joint 1 → cols 2,3) ──
    raw_kp  = raw.reshape(-1, NWU_NUM_JOINTS, 2)       # (N, 18, 2)
    aligned = np.delete(raw_kp, NWU_DROP_JOINT, axis=1)  # (N, 17, 2)

    # ── Normalise: center on mid-hip, scale by torso ──
    X = normalise_nwu_keypoints(aligned)               # (N, 34)

    # ── Labels ──
    label_col  = [c for c in all_cols
                  if 'class' in c.lower() and 'text' in c.lower()]
    if not label_col:
        label_col = [c for c in all_cols if 'class' in c.lower()]
    y = (df[label_col[0]].str.strip().str.lower() == 'stand').astype(np.int32).values

    print(f"[NWU] {len(X)} rows  |  sit={( y==0).sum()}  stand={(y==1).sum()}")
    return X, y

In [5]:
# ─────────────────────────────────────────────
# STEP 2 — NORMALISE NWU KEYPOINTS
# ─────────────────────────────────────────────
def normalise_nwu_keypoints(kp_17: np.ndarray) -> np.ndarray:
    """
    Normalise (N, 17, 2) aligned OpenPose keypoints.
    After dropping Neck, COCO 17 joint order applies.
    KP indices from shared_utils.KP are valid here.

    Centers on mid-hip, scales by torso (mid-hip → mid-shoulder).
    Returns (N, 34) flattened.
    """
    kp = kp_17.copy()

    # After dropping joint 1 (Neck), original joints 0,2-17
    # shift: new index = old index - 1 for joints > 1
    # Nose stays at 0; shoulders now at 4,5; hips at 10,11
    # These match KP["left_shoulder"]=5 etc. from shared_utils
    mid_hip = (kp[:, KP["left_hip"],      :] + kp[:, KP["right_hip"],      :]) / 2
    mid_sho = (kp[:, KP["left_shoulder"], :] + kp[:, KP["right_shoulder"], :]) / 2

    torso   = np.linalg.norm(mid_sho - mid_hip, axis=1, keepdims=True)
    torso   = np.clip(torso, 1e-6, None)

    kp     -= mid_hip[:, np.newaxis, :]
    kp     /= torso[:, np.newaxis, :]

    return kp.reshape(-1, 17 * 2).astype(np.float32)   # (N, 34)

In [6]:
# ─────────────────────────────────────────────
# STEP 3 — BUILD SYNTHETIC TEMPORAL WINDOWS
# ─────────────────────────────────────────────
def build_synthetic_windows(X: np.ndarray, y: np.ndarray,
                             seq_len: int = SEQUENCE_LEN,
                             augment: bool = True) -> tuple:
    """
    NWU contains static frames — not video sequences.
    We build synthetic temporal windows by grouping consecutive same-label
    frames into windows of length seq_len, simulating what the model sees
    during a sustained pose in real video.

    FIX 1 — Sit oversampling:
      The NWU dataset has a 2.9:1 stand:sit imbalance which carried through
      to the window distribution (862 sit vs 2514 stand), directly causing
      the Sit F1=0.83 gap.
      Solution: use a tighter sliding step for the minority sit class
      (seq_len//6 = 10 frames) vs the majority stand class (seq_len//2 = 30
      frames). This generates ~3× more sit windows, bringing the ratio near 1:1.

    Augmentation adds small Gaussian noise per window to improve robustness
    against natural pose variation in home monitoring conditions.

    Returns: X_win (M, seq_len, 34),  y_win (M,)
    """
    X_win, y_win = [], []

    # FIX 1: class-specific step sizes to balance window counts
    step_per_class = {
        0: seq_len // 6,   # sit   — tighter step → more windows (minority class)
        1: seq_len // 2,   # stand — normal 50% overlap (majority class)
    }

    for label in [0, 1]:
        idx    = np.where(y == label)[0]
        frames = X[idx]
        rng    = np.random.default_rng(seed=SEED)
        perm   = rng.permutation(len(frames))
        frames = frames[perm]
        step   = step_per_class[label]

        for start in range(0, len(frames) - seq_len + 1, step):
            window = frames[start: start + seq_len]
            X_win.append(window)
            y_win.append(label)

            if augment:
                noisy = window + np.random.normal(0, 0.01, window.shape).astype(np.float32)
                X_win.append(noisy)
                y_win.append(label)

    X_win = np.array(X_win, dtype=np.float32)   # (M, seq_len, 34)
    y_win = np.array(y_win, dtype=np.int32)

    sit_n   = (y_win == 0).sum()
    stand_n = (y_win == 1).sum()
    print(f"[5STS] Windows: {X_win.shape}  |  "
          f"sit={sit_n}  stand={stand_n}  "
          f"ratio={stand_n/sit_n:.2f}:1  (target ~1:1)")
    return X_win, y_win

In [7]:
# ─────────────────────────────────────────────
# STEP 4 — MODEL  Conv1D → GRU → Dense
# ─────────────────────────────────────────────
def build_model(input_dim: int = NWU_ALIGNED_DIM,
                seq_len: int   = SEQUENCE_LEN) -> keras.Model:
    """
    Conv1D → GRU → Dense(sigmoid)

    Conv1D : captures frame-to-frame sit/stand transitions (local patterns)
    GRU    : learns sustained posture patterns over the full window
    Dense  : outputs probability of 'standing' state

    Input shape: (batch, seq_len, input_dim) = (batch, 60, 34)
    """
    inputs = keras.Input(shape=(seq_len, input_dim), name="keypoint_sequence")

    # ── Conv1D block ──
    x = layers.Conv1D(64,  kernel_size=3, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    # ── GRU block ──
    x = layers.GRU(64, return_sequences=False)(x)
    x = layers.Dropout(0.3)(x)

    # ── Classification head ──
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="sit_stand")(x)

    model = keras.Model(inputs, outputs, name="SitStandModel")
    model.summary()
    return model

In [8]:
# ─────────────────────────────────────────────
# STEP 5 — TRAINING
# ─────────────────────────────────────────────
def train_model(model: keras.Model,
                X_train: np.ndarray, y_train: np.ndarray,
                X_val:   np.ndarray, y_val:   np.ndarray) -> keras.Model:
    """
    FIX 2 — Manual class weights {sit:3.0, stand:1.0}:
      Replaces the formula-derived weights (which gave ~2.9 sit, ~0.7 stand).
      The manual values are tuned to the observed F1 gap (Sit=0.83, Stand=0.94)
      rather than just the raw count ratio, pushing the model harder on sit.

    FIX 3 — Label smoothing (0.1):
      The train/val loss gap (0.14 vs 0.29 = 2×) indicated the model was
      overconfident on training examples. Label smoothing softens hard 0/1
      targets to 0.05/0.95, acting as regularisation and closing the gap.
    """
    # FIX 2: manually tuned weights targeting the F1 gap directly
    class_w = {0: 3.0, 1: 1.0}
    print(f"[5STS] Class weights — sit: {class_w[0]:.1f}  stand: {class_w[1]:.1f}")

    # FIX 3: label smoothing to reduce overconfidence
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss=keras.losses.BinaryCrossentropy(label_smoothing=0.1),
        metrics=["accuracy"],
    )

    # Generate timestamp
    timestamp = datetime.now().strftime("%y%m%d_%H%M%S")

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5,
            restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=3, verbose=1),
        keras.callbacks.ModelCheckpoint(
            f"../models/5sts_best_{timestamp}.keras", 
            monitor="val_loss",
            save_best_only=True, verbose=1),
    ]

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_w,
        callbacks=callbacks,
        verbose=1,
    )
    return model

In [9]:
# ─────────────────────────────────────────────
# STEP 6 — EVALUATION
# ─────────────────────────────────────────────
def evaluate(model: keras.Model,
             X_test: np.ndarray, y_test: np.ndarray):
    probs  = model.predict(X_test, batch_size=BATCH_SIZE).squeeze()
    preds  = (probs >= 0.5).astype(int)
    print("\n[5STS Evaluation]")
    print(classification_report(y_test, preds, target_names=["Sit", "Stand"]))

In [10]:
# ─────────────────────────────────────────────
# STEP 7 — EXPORT TO TFLITE
# ─────────────────────────────────────────────
def export_tflite(model: keras.Model,
                  output_path: str = "5sts_model.tflite"):
    """
    Export trained model to TensorFlow Lite for edge/mobile deployment.
    Enables home monitoring on Raspberry Pi, Android, or tablet.
    """
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]   # INT8 quantisation
    tflite_model = converter.convert()
    with open(output_path, "wb") as f:
        f.write(tflite_model)
    print(f"[5STS] TFLite model saved → {output_path}  "
          f"({os.path.getsize(output_path) / 1024:.1f} KB)")

In [11]:
# ─────────────────────────────────────────────
# STEP 8 — INFERENCE on patient video
# ─────────────────────────────────────────────
def infer_5sts(model: keras.Model, movenet_fn, input_size: int,
               video_path: str, fps: float = 30.0) -> dict:
    """
    Runs on a patient's home video recording of the 5STS test.
    MoveNet Lightning extracts 17 COCO keypoints per frame.
    State machine counts 10 sit↔stand transitions = 5 complete reps.
    """
    raw  = extract_keypoints_from_video(movenet_fn, input_size, video_path)
    norm = normalize_keypoints(raw)          # (T, 17, 2)
    flat = norm.reshape(len(norm), -1)       # (T, 34)

    preds = []
    step  = SEQUENCE_LEN // 2

    for s in range(0, len(flat) - SEQUENCE_LEN + 1, step):
        window = flat[s: s + SEQUENCE_LEN][np.newaxis]   # (1, 60, 34)
        prob   = model.predict(window, verbose=0)[0, 0]
        preds.append((s / fps, float(prob)))

    # State machine: 10 transitions = 5 complete sit→stand reps
    state, transitions = "unknown", 0
    start_time = end_time = None

    for t, prob in preds:
        # FIX 4: raised STAND threshold 0.60→0.65, lowered SIT threshold 0.40→0.35
        # Forces the model to be more certain before calling a transition,
        # reducing false sit detections that corrupt the rep count.
        if prob >= 0.65 and state != "standing":
            if state == "sitting":
                transitions += 1
            if transitions == 1 and start_time is None:
                start_time = t
            state = "standing"
        elif prob <= 0.35 and state != "sitting":
            if state == "standing":
                transitions += 1
            if transitions >= 10:
                end_time = t
                break
            state = "sitting"

    if start_time is None or end_time is None:
        return {"error": "Could not detect 5 complete cycles.",
                "transitions_detected": transitions}

    duration = round(end_time - start_time, 2)
    risk     = "High" if duration >= FALL_RISK_THRESHOLD else "Low"

    result = {
        "test"             : "5STS",
        "duration_seconds" : duration,
        "transitions"      : transitions,
        "fall_risk"        : risk,
        "threshold_seconds": FALL_RISK_THRESHOLD,
    }
    print(f"[5STS] {duration:.1f}s  →  Fall risk: {risk}")
    return result

In [12]:
# ─────────────────────────────────────────────
# STEP 9 — LONGITUDINAL COMPARISON
# ─────────────────────────────────────────────
def compare(baseline: dict, followup: dict) -> dict:
    """Flag worsening if duration increased >= 2s or risk moved Low → High."""
    d0, d1 = baseline["duration_seconds"], followup["duration_seconds"]
    delta   = round(d1 - d0, 2)
    worse   = (delta >= WORSEN_DELTA) or \
              (baseline["fall_risk"] == "Low" and followup["fall_risk"] == "High")

    report = {
        "test"         : "5STS",
        "baseline_s"   : d0,
        "followup_s"   : d1,
        "delta_s"      : delta,
        "worsened"     : worse,
        "baseline_risk": baseline["fall_risk"],
        "followup_risk": followup["fall_risk"],
        "action"       : "⚠️  Alert clinician" if worse else "✅  Stable",
    }
    print(json.dumps(report, indent=2))
    return report


In [13]:
# ── 1. Load NWU CSV ──────────────────────────────────────────────────
X, y = load_nwu_dataset(NWU_XLSX_PATH)

[NWU] Loading: ../nwu_human_pose_dataset/dataset_HumanPose_KeypointCoordinates.xlsx
[NWU] 50727 rows  |  sit=12979  stand=37748


In [14]:
# ── 2. Build synthetic temporal windows ──────────────────────────────
X_win, y_win = build_synthetic_windows(X, y, seq_len=SEQUENCE_LEN)

[5STS] Windows: (5098, 60, 34)  |  sit=2584  stand=2514  ratio=0.97:1  (target ~1:1)


In [15]:
# ── 3. Train / val / test split ──────────────────────────────────────
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X_win, y_win, test_size=0.2, stratify=y_win, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

In [16]:
# ── 4. Build & train model ───────────────────────────────────────────
model = build_model()
model = train_model(model, X_tr, y_tr, X_val, y_val)

Model: "SitStandModel"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ keypoint_sequence (InputLayer)  │ (None, 60, 34)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 60, 64)         │         6,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 60, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 60, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 60, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sit_stand (Dense)               │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 73,537 (287.25 KB)

 Trainable params: 73,153 (285.75 KB)

 Non-trainable params: 384 (1.50 KB)

[5STS] Class weights — sit: 3.0  stand: 1.0
Epoch 1/30
126/128 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5124 - loss: 1.2035
Epoch 1: val_loss improved from None to 0.75018, saving model to ../models/5sts_best_260302_160149.keras

Epoch 1: finished saving model to ../models/5sts_best_260302_160149.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - accuracy: 0.5128 - loss: 1.1829 - val_accuracy: 0.5098 - val_loss: 0.7502 - learning_rate: 0.0010
Epoch 2/30
127/128 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5248 - loss: 1.1560
Epoch 2: val_loss did not improve from 0.75018
128/128 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.5280 - loss: 1.1582 - val_accuracy: 0.5235 - val_loss: 0.7752 - learning_rate: 0.0010
Epoch 3/30
127/128 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5324 - loss: 1.1459
Epoch 3: val_loss did not improve from 0.75018
128/128 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.5370 - loss: 1.1441 - val_accuracy: 0.5373 - val_loss: 0.7844 - learning_rate: 0.

In [17]:
# ── 5. Evaluate on held-out test set ─────────────────────────────────
evaluate(model, X_test, y_test)

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

[5STS Evaluation]
              precision    recall  f1-score   support

         Sit       0.92      0.99      0.95       259
       Stand       0.99      0.91      0.95       251

    accuracy                           0.95       510
   macro avg       0.95      0.95      0.95       510
weighted avg       0.95      0.95      0.95       510



In [18]:
# ── 6. Export to TFLite for home device deployment ───────────────────
# export_tflite(model, "5sts_model.tflite")

# ── 7. Patient inference (uncomment with real video paths) ───────────
# movenet_fn, input_size = load_movenet("lightning")   # fast for home use
# baseline = infer_5sts(model, movenet_fn, input_size, "patient_baseline.mp4")
# followup = infer_5sts(model, movenet_fn, input_size, "patient_week8.mp4")
# compare(baseline, followup)